# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dipson-mishra/flyrank-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import os
from pathlib import Path
import duckdb
import pandas as pd

def load_token():
    for path in [Path.cwd() / '.env', Path.cwd().parent / '.env', Path.cwd().parent.parent / '.env']:
        if path.exists():
            for line in path.read_text().splitlines():
                if line.strip().startswith('HF_TOKEN='):
                    return line.split('=', 1)[1].strip().strip(chr(34)).strip(chr(39))
    return os.environ.get('HF_TOKEN')

token = load_token()
if not token:
    raise RuntimeError('HF_TOKEN is required through .env or the environment.')
con = duckdb.connect()
con.execute('CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN ?)', [token])
FACT = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
CONTENT = 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
print('HF warehouse connection configured; token value is not displayed.')

HF warehouse connection configured; token value is not displayed.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [2]:
query = f"""
WITH daily AS (
  SELECT content_hash_id, client_hash_id,
         SUM(COALESCE(gsc_impressions, 0)) AS impressions,
         SUM(COALESCE(gsc_clicks, 0)) AS clicks,
         SUM(COALESCE(gsc_sum_position, 0)) AS sum_position,
         SUM(COALESCE(ga4_sessions, 0)) AS sessions,
         SUM(COALESCE(ga4_engaged_sessions, 0)) AS engaged_sessions
  FROM read_parquet('{FACT}')
  WHERE gsc_data_available IS TRUE
  GROUP BY content_hash_id, client_hash_id
)
SELECT d.*, c.content_type, c.main_intent, c.word_count,
       d.clicks * 100.0 / NULLIF(d.impressions, 0) AS ctr,
       d.sum_position * 1.0 / NULLIF(d.impressions, 0) AS avg_position
FROM daily d
LEFT JOIN read_parquet('{CONTENT}') c USING (content_hash_id)
WHERE d.impressions > 0
"""
frame = con.sql(query).df()
frame['position_tier'] = pd.cut(frame['avg_position'], bins=[0, 3, 10, 20, 50, float('inf')], labels=['top_3', 'page_1', 'striking', 'page_3_5', 'deep'], right=True).astype('string').fillna('no_data')
print(f'Aggregated content-client rows: {len(frame):,}')
print(f'Unique content items: {frame.content_hash_id.nunique():,}')
print(f'Rows at or above the 500-impression development floor: {(frame.impressions >= 500).sum():,}')
print(frame[['impressions', 'clicks', 'sessions', 'ctr', 'avg_position']].describe(percentiles=[.5, .9, .99]).round(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Aggregated content-client rows: 176,738
Unique content items: 176,738
Rows at or above the 500-impression development floor: 61,924
       impressions      clicks    sessions         ctr  avg_position
count   176738.000  176738.000  176738.000  176738.000    176738.000
mean      1587.987       4.650       7.009       0.459        15.992
std       5431.338      26.723      34.205       3.776        18.098
min          1.000       0.000       0.000       0.000         0.000
50%        173.000       0.000       0.000       0.000         8.178
90%       3930.000      10.000      12.000       0.615        41.053
99%      21799.780      73.000     129.630       5.882        80.167
max     617124.000    5668.000    2603.000     100.000       309.000


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [3]:
eligible = frame[(frame.impressions >= 500) & (frame.position_tier != 'no_data')].copy()
eligible['impression_bin'] = pd.qcut(eligible['impressions'], q=4, duplicates='drop')
volume_test = eligible.groupby('impression_bin', observed=True).agg(n=('content_hash_id', 'size'), impressions=('impressions', 'sum'), clicks=('clicks', 'sum'))
volume_test['weighted_ctr'] = 100 * volume_test['clicks'] / volume_test['impressions']
print('Signal 1 — volume quartiles and CTR:')
print(volume_test.round(3))
print('Verdict:', 'CONFIRMED' if volume_test['weighted_ctr'].is_monotonic_increasing else 'MIXED', '- volume is primarily an evidence-strength guardrail.')

position_test = eligible.groupby('position_tier', observed=True).agg(n=('content_hash_id', 'size'), impressions=('impressions', 'sum'), clicks=('clicks', 'sum'))
position_test['weighted_ctr'] = 100 * position_test['clicks'] / position_test['impressions']
print('\nSignal 2 — position tiers and CTR:')
print(position_test.round(3))
ordered = position_test.reindex(['top_3', 'page_1', 'striking', 'page_3_5', 'deep']).dropna()
print('Verdict:', 'CONFIRMED' if len(ordered) >= 2 and ordered['weighted_ctr'].is_monotonic_decreasing else 'MIXED', '- position is used for adjustment, not as a causal claim.')

type_test = eligible.groupby(['position_tier', 'content_type'], dropna=False, observed=True).agg(n=('content_hash_id', 'size'), impressions=('impressions', 'sum'), clicks=('clicks', 'sum'))
type_test['weighted_ctr'] = 100 * type_test['clicks'] / type_test['impressions']
print('\nSignal 3 — content type within position cells with n >= 50:')
print(type_test[type_test['n'] >= 50].round(3).head(30))
print('Verdict: MIXED unless stable differences persist in adequately sized cells.')

Signal 1 — volume quartiles and CTR:
                         n  impressions    clicks  weighted_ctr
impression_bin                                                 
(499.999, 940.0]     15500   10762882.0   27040.0         0.251
(940.0, 1898.0]      15464   20928342.0   58800.0         0.281
(1898.0, 4543.25]    15479   45745761.0  137671.0         0.301
(4543.25, 617124.0]  15481  191479134.0  569238.0         0.297
Verdict: MIXED - volume is primarily an evidence-strength guardrail.

Signal 2 — position tiers and CTR:
                   n  impressions    clicks  weighted_ctr
position_tier                                            
deep             682    1343204.0     452.0         0.034
page_1         32300  143336395.0  464788.0         0.324
page_3_5       10468   54973168.0   76829.0         0.140
striking       10534   28550734.0   92636.0         0.324
top_3           7940   40712618.0  158044.0         0.388
Verdict: MIXED - position is used for adjustment, not as a causal cl

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [4]:
eligible['below_position_median'] = eligible['ctr'] < eligible.groupby('position_tier', observed=True)['ctr'].transform('median')
flag_test = eligible.groupby('below_position_median', observed=True).agg(n=('content_hash_id', 'size'), impressions=('impressions', 'sum'), median_ctr=('ctr', 'median'))
print('Flag-linked test — below-median CTR within a valid position tier:')
print(flag_test.round(3))
assert eligible['impressions'].ge(500).all()
assert eligible['position_tier'].ne('no_data').all()
print('The measured rule supports a review queue for sufficiently observed, visible pages; it does not establish that an edit will cause recovery.')

Flag-linked test — below-median CTR within a valid position tier:
                           n  impressions  median_ctr
below_position_median                                
False                  31308  146416465.0       0.381
True                   30616  122499654.0       0.067
The measured rule supports a review queue for sufficiently observed, visible pages; it does not establish that an edit will cause recovery.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.